<a href="https://colab.research.google.com/github/ArghyaRC96/metricguard-ai/blob/main/notebooks/03_chunk_metadata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ MetricGuard AI — Source-Aware Chunking and Metadata

## Phase 3 — Chunking and Metadata Enrichment

This notebook converts canonical parsed documents into retrieval-ready
knowledge chunks.

Chunking strategy varies by source type:

- Markdown → heading-aware
- SQL → SQL-aware recursive splitting
- JSON/YAML → structure-aware splitting
- CSV → dataset summaries preserved where practical

Each chunk receives deterministic identifiers and metadata for later
retrieval, filtering, version analysis, freshness checks, lineage,
reranking, and citation generation.

In [1]:
from pathlib import Path
import shutil
import subprocess

GITHUB_USERNAME = "ArghyaRC96"

REPO_URL = f"https://github.com/{GITHUB_USERNAME}/metricguard-ai.git"
REPO_DIR = Path("/content/metricguard-ai")

In [2]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", REPO_URL, str(REPO_DIR)],
    check=True,
)

print("Repository cloned:", REPO_DIR)

Repository cloned: /content/metricguard-ai


In [3]:
%pip install -q -e "/content/metricguard-ai" langchain-text-splitters

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for metricguard-ai (pyproject.toml) ... done


In [6]:
%pip install -q -e "/content/metricguard-ai"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for metricguard-ai (pyproject.toml) ... done


In [7]:
import sys
sys.path.append("/content/metricguard-ai/src")

In [8]:
from pathlib import Path
import json
import re
from typing import Any

import pandas as pd
import yaml

from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

from metricguard.ingestion import parse_knowledge_base

### Defining project paths

In [9]:
RAW_DIR = REPO_DIR / "data" / "raw"
PROCESSED_DIR = REPO_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Raw:", RAW_DIR)
print("Processed:", PROCESSED_DIR)

Raw: /content/metricguard-ai/data/raw
Processed: /content/metricguard-ai/data/processed


### Loading project config

In [10]:
CONFIG_PATH = REPO_DIR / "configs" / "settings.yaml"

with CONFIG_PATH.open(
    "r",
    encoding="utf-8-sig",
) as file:
    config = yaml.safe_load(file)

config

{'project': {'name': 'metricguard-ai',
  'version': '0.1.0',
  'environment': 'development'},
 'retrieval': {'top_k': 5, 'rerank_enabled': True},
 'chunking': {'strategy': 'recursive',
  'chunk_size': 800,
  'chunk_overlap': 120},
 'embeddings': {'model': None},
 'vector_database': {'provider': 'qdrant', 'collection_name': 'metricguard'},
 'confidence': {'minimum_threshold': 0.6},
 'security': {'authentication_enabled': True,
  'role_based_access_control': True,
  'default_role': 'viewer'},
 'logging': {'level': 'INFO', 'audit_logging_enabled': True}}

In [11]:
CHUNK_SIZE = config["chunking"]["chunk_size"]
CHUNK_OVERLAP = config["chunking"]["chunk_overlap"]

print("Chunk size:", CHUNK_SIZE)
print("Chunk overlap:", CHUNK_OVERLAP)

Chunk size: 800
Chunk overlap: 120


In [12]:
parsed_documents, parse_errors = parse_knowledge_base(
    source_root=RAW_DIR,
    repo_root=REPO_DIR,
)

print("Documents:", len(parsed_documents))
print("Errors:", len(parse_errors))

Documents: 55
Errors: 0


In [13]:
parse_errors

[]

In [14]:
parsed_documents[0]

ParsedDocument(document_id='active_customer_review-1f2c34aa6c80', source_path='data/raw/analyst_notes/active_customer_review.md', file_name='active_customer_review.md', source_type='markdown', content_hash='1f2c34aa6c80438913880b4b4a3aab3fc5d5357eac702f6be42c79e4a314bb2e', content='# Active Customer Definition Review\n\nauthor: Rohan Mehta\nteam: Customer Analytics\ndate: 2026-03-10\nrelated_metric: active_customers\n\nThe enterprise Active Customer definition changed on March 1, 2026.\n\nThe approved definition now requires at least one successfully paid order\nduring the previous 30 days.\n\nGrowth reporting currently continues to count identified customers with recent\ndigital activity.\n\nThat measure remains useful for engagement analysis but should not be treated\nas equivalent to the enterprise Active Customers KPI.\n\nRecommendation:\n\nEither migrate the Growth dashboard to Active Customers v2 or rename the\nexisting measure to Digital Active Customers.\n', structured_data=Non

### Fallback Splitter

In [15]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        "",
    ],
)

### Markdown Splitter

In [16]:
markdown_header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "header_1"),
        ("##", "header_2"),
        ("###", "header_3"),
    ],
    strip_headers=False,
)

In [17]:
def split_if_needed(text: str) -> list[str]:
    """
    Keep short logical sections intact.

    Only recursively split when a section exceeds
    the configured chunk size.
    """

    text = text.strip()

    if not text:
        return []

    if len(text) <= CHUNK_SIZE:
        return [text]

    return recursive_splitter.split_text(text)

### Markdown Chunker

In [18]:
def chunk_markdown(content: str) -> list[dict[str, Any]]:
    sections = markdown_header_splitter.split_text(content)

    chunks = []

    for section in sections:

        section_metadata = dict(section.metadata)

        for text in split_if_needed(section.page_content):

            chunks.append(
                {
                    "content": text,
                    "section_metadata": section_metadata,
                }
            )

    return chunks

### SQL-aware chunker

In [19]:
sql_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=[
        "\n\n",
        "\nWITH ",
        "\nSELECT ",
        "\nFROM ",
        "\nJOIN ",
        "\nLEFT JOIN ",
        "\nINNER JOIN ",
        "\nWHERE ",
        "\nGROUP BY ",
        "\nORDER BY ",
        "\nHAVING ",
        "\n",
        " ",
        "",
    ],
)

In [20]:
def chunk_sql(content: str) -> list[dict[str, Any]]:
    content = content.strip()

    if len(content) <= CHUNK_SIZE:

        return [
            {
                "content": content,
                "section_metadata": {},
            }
        ]

    return [
        {
            "content": text,
            "section_metadata": {},
        }
        for text in sql_splitter.split_text(content)
    ]

### JSON/YAML Chunking

In [21]:
def chunk_structured_data(
    structured_data: Any,
    original_content: str,
) -> list[dict[str, Any]]:

    if not structured_data:
        return [
            {
                "content": text,
                "section_metadata": {},
            }
            for text in split_if_needed(original_content)
        ]

    chunks = []

    if isinstance(structured_data, dict):

        for key, value in structured_data.items():

            # Lists such as models, metrics, sources, etc.
            if isinstance(value, list):

                for index, item in enumerate(value):

                    serialized = json.dumps(
                        {
                            key: item
                        },
                        indent=2,
                        ensure_ascii=False,
                        default=str,
                    )

                    for text in split_if_needed(serialized):

                        chunks.append(
                            {
                                "content": text,
                                "section_metadata": {
                                    "structure_key": key,
                                    "structure_index": index,
                                },
                            }
                        )

            else:

                serialized = json.dumps(
                    {
                        key: value
                    },
                    indent=2,
                    ensure_ascii=False,
                    default=str,
                )

                for text in split_if_needed(serialized):

                    chunks.append(
                        {
                            "content": text,
                            "section_metadata": {
                                "structure_key": key,
                            },
                        }
                    )

    else:

        for text in split_if_needed(original_content):

            chunks.append(
                {
                    "content": text,
                    "section_metadata": {},
                }
            )

    return chunks

### CSV Chunking


In [22]:
def chunk_csv(content: str) -> list[dict[str, Any]]:

    return [
        {
            "content": text,
            "section_metadata": {},
        }
        for text in split_if_needed(content)
    ]

### Creating Chunk Dispatcher

In [23]:
def chunk_document(document) -> list[dict[str, Any]]:

    source_type = document.source_type

    if source_type == "markdown":
        return chunk_markdown(
            document.content
        )

    if source_type == "sql":
        return chunk_sql(
            document.content
        )

    if source_type in {"json", "yaml"}:
        return chunk_structured_data(
            structured_data=document.structured_data,
            original_content=document.content,
        )

    if source_type == "csv":
        return chunk_csv(
            document.content
        )

    raise ValueError(
        f"Unsupported source type: {source_type}"
    )

### Defining Asset Category

In [24]:
def infer_asset_type(source_path: str) -> str:

    path = source_path.replace("\\", "/").lower()

    if "/business_rules/" in path:
        return "business_rule"

    if "/sql/staging/" in path:
        return "staging_model"

    if "/sql/facts/" in path:
        return "fact_model"

    if "/sql/marts/" in path:
        return "mart"

    if "/sql/metrics/" in path:
        return "metric_sql"

    if "/dashboards/" in path:
        return "dashboard"

    if "/dbt/" in path:
        return "dbt_documentation"

    if "/incidents/" in path:
        return "incident"

    if "/analyst_notes/" in path:
        return "analyst_note"

    if "/tabular/" in path:
        return "raw_dataset"

    return "unknown"

In [25]:
VERSIONED_METRIC_PATTERN = re.compile(
    r"^(?P<metric_name>.+)_"
    r"(?P<version>v\d+)$"
)


def infer_metric_version(file_name: str) -> dict[str, Any]:

    stem = Path(file_name).stem

    match = VERSIONED_METRIC_PATTERN.match(stem)

    if not match:
        return {
            "metric_name": None,
            "version": None,
        }

    return {
        "metric_name": match.group("metric_name"),
        "version": match.group("version"),
    }

In [26]:
infer_metric_version(
    "net_revenue_v3.md"
)

{'metric_name': 'net_revenue', 'version': 'v3'}

In [27]:
infer_metric_version(
    "net_revenue_v3.md"
)

{'metric_name': 'net_revenue', 'version': 'v3'}

### Extracting JSON dashboard metadata

In [28]:
def extract_dashboard_metadata(
    document,
) -> dict[str, Any]:

    if document.source_type != "json":
        return {}

    data = document.structured_data

    if not isinstance(data, dict):
        return {}

    metadata = {}

    for key in [
        "dashboard_id",
        "dashboard_name",
        "owner_team",
        "business_domain",
        "status",
        "last_reviewed",
        "refresh_frequency",
        "source_mart",
    ]:

        if key in data:
            metadata[key] = data[key]

    if isinstance(data.get("metrics"), list):

        metadata["metric_names"] = [
            metric.get("metric_name")
            for metric in data["metrics"]
            if metric.get("metric_name")
        ]

        metadata["metric_versions"] = {
            metric["metric_name"]: metric.get(
                "metric_version"
            )
            for metric in data["metrics"]
            if metric.get("metric_name")
        }

    return metadata

In [29]:
MARKDOWN_METADATA_FIELDS = {
    "incident_id",
    "status",
    "severity",
    "opened_date",
    "closed_date",
    "owner",
    "author",
    "team",
    "date",
    "related_metric",
    "affected_metric",
}


def extract_markdown_metadata(
    content: str,
) -> dict[str, Any]:

    metadata = {}

    for line in content.splitlines():

        if ":" not in line:
            continue

        key, value = line.split(
            ":",
            1,
        )

        key = key.strip().lower()
        value = value.strip()

        if (
            key in MARKDOWN_METADATA_FIELDS
            and value
        ):
            metadata[key] = value

    return metadata

### Building base document metadata

In [30]:
def build_document_metadata(
    document,
) -> dict[str, Any]:

    metadata = {
        "document_id": document.document_id,
        "source_path": document.source_path,
        "file_name": document.file_name,
        "source_type": document.source_type,
        "asset_type": infer_asset_type(
            document.source_path
        ),
        "content_hash": document.content_hash,
    }

    metric_metadata = infer_metric_version(
        document.file_name
    )

    metadata.update(metric_metadata)

    if document.source_type == "json":

        metadata.update(
            extract_dashboard_metadata(document)
        )

    if document.source_type == "markdown":

        metadata.update(
            extract_markdown_metadata(
                document.content
            )
        )

    return metadata

In [31]:
net_revenue_doc = next(
    doc
    for doc in parsed_documents
    if doc.file_name == "net_revenue_v3.md"
)

In [32]:
build_document_metadata(
    net_revenue_doc
)

{'document_id': 'net_revenue_v3-da452b3b70f7',
 'source_path': 'data/raw/business_rules/net_revenue/net_revenue_v3.md',
 'file_name': 'net_revenue_v3.md',
 'source_type': 'markdown',
 'asset_type': 'business_rule',
 'content_hash': 'da452b3b70f71dd93224b10b26194b4f2ffc40e4644d35127628ee799912561b',
 'metric_name': 'net_revenue',
 'version': 'v3',
 'status': 'active',
 'owner': 'Finance Analytics'}

### Building final chunk objects

In [33]:
def build_chunks(document) -> list[dict[str, Any]]:

    raw_chunks = chunk_document(document)

    base_metadata = build_document_metadata(
        document
    )

    final_chunks = []

    total_chunks = len(raw_chunks)

    for index, chunk in enumerate(raw_chunks):

        chunk_id = (
            f"{document.document_id}"
            f"-chunk-{index:04d}"
        )

        metadata = {
            **base_metadata,
            **chunk["section_metadata"],
            "chunk_index": index,
            "chunk_count": total_chunks,
        }

        final_chunks.append(
            {
                "chunk_id": chunk_id,
                "content": chunk["content"],
                "metadata": metadata,
            }
        )

    return final_chunks

### Chunking Complete Knowledge Base

In [34]:
all_chunks = []

chunk_errors = []

for document in parsed_documents:

    try:

        chunks = build_chunks(document)

        all_chunks.extend(chunks)

    except Exception as exc:

        chunk_errors.append(
            {
                "document": document.source_path,
                "error": str(exc),
            }
        )

print(
    "Source documents:",
    len(parsed_documents)
)

print(
    "Chunks generated:",
    len(all_chunks)
)

print(
    "Chunk errors:",
    len(chunk_errors)
)

Source documents: 55
Chunks generated: 167
Chunk errors: 0


In [36]:
chunk_errors

[]

In [37]:
all_chunks[0]

{'chunk_id': 'active_customer_review-1f2c34aa6c80-chunk-0000',
 'content': '# Active Customer Definition Review  \nauthor: Rohan Mehta\nteam: Customer Analytics\ndate: 2026-03-10\nrelated_metric: active_customers  \nThe enterprise Active Customer definition changed on March 1, 2026.  \nThe approved definition now requires at least one successfully paid order\nduring the previous 30 days.  \nGrowth reporting currently continues to count identified customers with recent\ndigital activity.  \nThat measure remains useful for engagement analysis but should not be treated\nas equivalent to the enterprise Active Customers KPI.  \nRecommendation:  \nEither migrate the Growth dashboard to Active Customers v2 or rename the\nexisting measure to Digital Active Customers.',
 'metadata': {'document_id': 'active_customer_review-1f2c34aa6c80',
  'source_path': 'data/raw/analyst_notes/active_customer_review.md',
  'file_name': 'active_customer_review.md',
  'source_type': 'markdown',
  'asset_type': 'a

In [38]:
[
    chunk
    for chunk in all_chunks
    if chunk["metadata"].get(
        "metric_name"
    ) == "net_revenue"
][:3]

[{'chunk_id': 'net_revenue_v1-3cbb74103fe4-chunk-0000',
  'content': '# Net Revenue — Version 1  \nmetric_name: net_revenue\nversion: v1\nstatus: deprecated\nowner: Finance Analytics\neffective_from: 2025-01-01\neffective_to: 2025-08-31\nsuperseded_by: v2',
  'metadata': {'document_id': 'net_revenue_v1-3cbb74103fe4',
   'source_path': 'data/raw/business_rules/net_revenue/net_revenue_v1.md',
   'file_name': 'net_revenue_v1.md',
   'source_type': 'markdown',
   'asset_type': 'business_rule',
   'content_hash': '3cbb74103fe4e9a3177a1214ec6a0dea0f79137fc41c82b0783ae610d8295b03',
   'metric_name': 'net_revenue',
   'version': 'v1',
   'status': 'deprecated',
   'owner': 'Finance Analytics',
   'header_1': 'Net Revenue — Version 1',
   'chunk_index': 0,
   'chunk_count': 3}},
 {'chunk_id': 'net_revenue_v1-3cbb74103fe4-chunk-0001',
  'content': '## Definition  \nNet Revenue is calculated as:  \nGross Revenue - Completed Refunds',
  'metadata': {'document_id': 'net_revenue_v1-3cbb74103fe4',
  

### Building the Chunk Dataframe

In [39]:
chunk_summary = pd.DataFrame(
    [
        {
            "chunk_id": chunk["chunk_id"],
            "source_type": chunk["metadata"]["source_type"],
            "asset_type": chunk["metadata"]["asset_type"],
            "file_name": chunk["metadata"]["file_name"],
            "metric_name": chunk["metadata"].get(
                "metric_name"
            ),
            "version": chunk["metadata"].get(
                "version"
            ),
            "content_length": len(
                chunk["content"]
            ),
        }
        for chunk in all_chunks
    ]
)

chunk_summary.head(10)

,chunk_id,source_type,asset_type,file_name,metric_name,version,content_length
0,active_customer_review-1f2c34aa6c80-chunk-0000,markdown,analyst_note,active_customer_review.md,None,None,679
1,conversion_rate_migration-71ee7fd90aa8-chunk-0000,markdown,analyst_note,conversion_rate_migration.md,None,None,559
2,revenue_v3_migration-d802bb74c774-chunk-0000,markdown,analyst_note,revenue_v3_migration.md,None,None,702
3,total_orders_semantics-92e0cd8e8726-chunk-0000,markdown,analyst_note,total_orders_semantics.md,None,None,598
4,active_customers_v1-d3ddd1ffc374-chunk-0000,markdown,business_rule,active_customers_v1.md,active_customers,v1,187
5,active_customers_v1-d3ddd1ffc374-chunk-0001,markdown,business_rule,active_customers_v1.md,active_customers,v1,121
6,active_customers_v1-d3ddd1ffc374-chunk-0002,markdown,business_rule,active_customers_v1.md,active_customers,v1,98
7,active_customers_v2-f900d6e1651c-chunk-0000,markdown,business_rule,active_customers_v2.md,active_customers,v2,157
8,active_customers_v2-f900d6e1651c-chunk-0001,markdown,business_rule,active_customers_v2.md,active_customers,v2,119
9,active_customers_v2-f900d6e1651c-chunk-0002,markdown,business_rule,active_customers_v2.md,active_customers,v2,131


### Verifying Chunk Distibution

In [40]:
chunk_summary.groupby(
    "source_type"
).agg(
    chunks=("chunk_id", "count"),
    average_length=(
        "content_length",
        "mean"
    ),
    maximum_length=(
        "content_length",
        "max"
    ),
).round(2)

,chunks,average_length,maximum_length
source_type,,,
csv,13,481.85,795
json,46,71.61,169
markdown,55,198.02,702
sql,31,462.90,798
yaml,22,234.27,779


In [42]:
assert all(
    chunk["content"].strip()
    for chunk in all_chunks
)

print("✅ No empty chunks.")

✅ No empty chunks.


In [43]:
chunk_ids = [
    chunk["chunk_id"]
    for chunk in all_chunks
]

assert len(chunk_ids) == len(
    set(chunk_ids)
)

print("✅ Chunk IDs are unique.")

✅ Chunk IDs are unique.


In [44]:
assert not any(
    "ground_truth"
    in chunk["metadata"]["source_path"]
    for chunk in all_chunks
)

print("✅ No ground-truth leakage.")

✅ No ground-truth leakage.


In [45]:
required_metadata = {
    "document_id",
    "source_path",
    "file_name",
    "source_type",
    "asset_type",
    "content_hash",
    "chunk_index",
    "chunk_count",
}


for chunk in all_chunks:

    missing = (
        required_metadata
        - set(chunk["metadata"])
    )

    assert not missing, (
        f"Missing metadata: {missing}"
    )

print(
    "✅ Required provenance metadata present."
)

✅ Required provenance metadata present.


In [47]:
versioned_chunks = [
    chunk
    for chunk in all_chunks
    if chunk["metadata"].get(
        "version"
    )
]

len(versioned_chunks)

43

In [48]:
[
    {
        "file": chunk["metadata"]["file_name"],
        "metric": chunk["metadata"]["metric_name"],
        "version": chunk["metadata"]["version"],
    }
    for chunk in versioned_chunks[:10]
]

[{'file': 'active_customers_v1.md',
  'metric': 'active_customers',
  'version': 'v1'},
 {'file': 'active_customers_v1.md',
  'metric': 'active_customers',
  'version': 'v1'},
 {'file': 'active_customers_v1.md',
  'metric': 'active_customers',
  'version': 'v1'},
 {'file': 'active_customers_v2.md',
  'metric': 'active_customers',
  'version': 'v2'},
 {'file': 'active_customers_v2.md',
  'metric': 'active_customers',
  'version': 'v2'},
 {'file': 'active_customers_v2.md',
  'metric': 'active_customers',
  'version': 'v2'},
 {'file': 'conversion_rate_v1.md',
  'metric': 'conversion_rate',
  'version': 'v1'},
 {'file': 'conversion_rate_v1.md',
  'metric': 'conversion_rate',
  'version': 'v1'},
 {'file': 'conversion_rate_v1.md',
  'metric': 'conversion_rate',
  'version': 'v1'},
 {'file': 'conversion_rate_v2.md',
  'metric': 'conversion_rate',
  'version': 'v2'}]

In [49]:
chunk_summary[
    chunk_summary["content_length"]
    > CHUNK_SIZE * 1.25
].sort_values(
    "content_length",
    ascending=False,
)

,chunk_id,source_type,asset_type,file_name,metric_name,version,content_length


### Writing chunks to JSONL

In [50]:
CHUNK_OUTPUT = (
    PROCESSED_DIR
    / "chunks.jsonl"
)

with CHUNK_OUTPUT.open(
    "w",
    encoding="utf-8",
) as file:

    for chunk in all_chunks:

        file.write(
            json.dumps(
                chunk,
                ensure_ascii=False,
                default=str,
            )
            + "\n"
        )

print(
    "Saved:",
    CHUNK_OUTPUT
)

Saved: /content/metricguard-ai/data/processed/chunks.jsonl


In [51]:
CHUNK_MANIFEST = (
    PROCESSED_DIR
    / "chunk_manifest.csv"
)

chunk_summary.to_csv(
    CHUNK_MANIFEST,
    index=False,
)

print(
    "Saved:",
    CHUNK_MANIFEST
)

Saved: /content/metricguard-ai/data/processed/chunk_manifest.csv


### Final Report

In [52]:
print("=" * 65)
print("METRICGUARD CHUNKING REPORT")
print("=" * 65)

print(
    f"Documents processed : {len(parsed_documents)}"
)

print(
    f"Chunks generated    : {len(all_chunks)}"
)

print(
    f"Chunking errors     : {len(chunk_errors)}"
)

print(
    f"Chunk size target   : {CHUNK_SIZE}"
)

print(
    f"Chunk overlap       : {CHUNK_OVERLAP}"
)

print(
    "Ground truth       : excluded"
)

print(
    "Metadata           : attached"
)

METRICGUARD CHUNKING REPORT
Documents processed : 55
Chunks generated    : 167
Chunking errors     : 0
Chunk size target   : 800
Chunk overlap       : 120
Ground truth       : excluded
Metadata           : attached
